# Kenya Conversational Engine: model weights export (run once)

Downloads the pinned model weights the GPU worker needs (Chatterbox base, the Sauti V2 adapter and Qwen3.5-4B, about **13 GB**) into this notebook's output as a `kenya-engine-weights/` bundle with a `MANIFEST.json`.

Attach that output to `kenya_engine_worker.ipynb` with **Add Input**. The worker then loads the weights from `/kaggle/input` in seconds instead of downloading 13 GB every session.

**How to run**
1. Settings → Accelerator **None** (CPU; this uses no GPU hours), Internet **On**. Optionally attach the `HF_TOKEN` secret (none of these repos is gated, but a token avoids anonymous rate limits).
2. **Save Version → Save & Run All (Commit)**. The run must *finish* for Kaggle to keep its output. That is why this is a separate notebook: the worker notebook never finishes.
3. When the version completes, attach it to the worker: in the worker notebook, **Add Input → Your Work → Notebooks → this notebook**. Optionally, from this version's **Output** tab, make it a private Dataset instead.

Re-run this only when the pinned revisions change. The worker ignores a bundle whose revisions don't match its own and downloads instead.

In [ ]:
# Cell 1: environment
import os
import shutil

try:
    from kaggle_secrets import UserSecretsClient

    _token = UserSecretsClient().get_secret("HF_TOKEN")
    if _token and _token.strip():
        os.environ["HF_TOKEN"] = _token.strip()  # never printed
        print("HF_TOKEN attached")
except Exception:
    print("HF_TOKEN not attached; downloading anonymously (fine for these public repos)")

OUT = "/kaggle/working/kenya-engine-weights"
free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"Free space in /kaggle/working: {free_gb:.1f} GB (the bundle needs ~13 GB; Kaggle keeps up to 20 GB of output)")
if free_gb < 15:
    raise RuntimeError("Not enough free space in /kaggle/working for the ~13 GB bundle.")

In [ ]:
# Cell 2: download the pinned weights and write the manifest
# Keep in sync with Cell 3/3b of kenya_engine_worker.ipynb.
import json
import shutil
import time
from pathlib import Path

from huggingface_hub import snapshot_download

SAUTI_TOKENIZER_FILE = "grapheme_mtl_merged_expanded_v1.json"
WEIGHTS_BUNDLE = "kenya-engine-weights"
WEIGHTS = {
    "chatterbox": {
        "repo": "ResembleAI/chatterbox",
        "revision": "5bb1f6ee58e50c3b8d408bc82a6d3740c2db6e18",
        "allow_patterns": ["ve.safetensors", "t3_cfg.safetensors", "s3gen.safetensors", "conds.pt", SAUTI_TOKENIZER_FILE],
    },
    "sauti": {
        "repo": "msingiai/sauti_tts_v2",
        "revision": "09de6d51288c5bf42931c082bcc9ebcbce68ab87",
        "allow_patterns": ["new_lang_adapter/*"],
    },
    "qwen": {
        "repo": "Qwen/Qwen3.5-4B",
        "revision": "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",
        "allow_patterns": ["*.json", "*.safetensors", "*.jinja", "*.txt"],
    },
}


def required_weight_files(name: str, root: Path) -> list:
    if name == "chatterbox":
        return [root / f for f in WEIGHTS["chatterbox"]["allow_patterns"]]
    if name == "sauti":
        return [root / "new_lang_adapter" / f for f in ("adapter_config.json", "adapter_model.safetensors")]
    index = root / "model.safetensors.index.json"
    files = [root / "config.json", root / "tokenizer_config.json", root / "chat_template.jinja", index]
    if index.is_file():
        files += [root / shard for shard in sorted(set(json.loads(index.read_text())["weight_map"].values()))]
    return files


manifest = {"bundle": WEIGHTS_BUNDLE, "created": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "models": {}}
for name, spec in WEIGHTS.items():
    target = Path(OUT) / name
    started = time.perf_counter()
    print(f"Downloading {spec['repo']}@{spec['revision'][:7]} -> {target}")
    snapshot_download(spec["repo"], revision=spec["revision"], allow_patterns=spec["allow_patterns"], local_dir=str(target))
    shutil.rmtree(target / ".cache", ignore_errors=True)  # huggingface_hub's download metadata, not needed
    missing = [str(p.relative_to(target)) for p in required_weight_files(name, target) if not p.is_file()]
    if missing:
        raise RuntimeError(f"{name}: download incomplete, missing {missing}")
    files = sorted(str(p.relative_to(target)).replace("\\", "/") for p in target.rglob("*") if p.is_file())
    size_gb = sum((target / f).stat().st_size for f in files) / 1e9
    manifest["models"][name] = {"repo": spec["repo"], "revision": spec["revision"], "files": files, "size_gb": round(size_gb, 2)}
    print(f"  {len(files)} files, {size_gb:.2f} GB in {time.perf_counter() - started:.0f}s")

(Path(OUT) / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))
print(f"\nBundle ready: {OUT} ({sum(m['size_gb'] for m in manifest['models'].values()):.2f} GB)")

In [ ]:
# Cell 3: next steps
print(
    "Done. This version's output now holds kenya-engine-weights/ (with MANIFEST.json).\n"
    "Open kenya_engine_worker.ipynb → Add Input → Your Work → Notebooks → select this notebook → Add.\n"
    "Its Cell 3b will report 'Weights bundle attached at /kaggle/input/...' and load the models from there."
)